In [ ]:
"""
    1. ted_id - TED domain identifier in the format AF-<UniProtID>-F1-model_v4_TED<domain_number_in_chain> i.e. AF-A0A1V6M2Y0-F1-model_v4_TED03
    2. md5_domain - md5 hash of domain sequence
    3. consensus_level - medium (2 methods agreement) or high (3 methods agreement)
    4. chopping - domain boundaries in the format <start>-<stop> or <start>-<stop>_<start>-<stop> for discontinuous domains
    5. nres_domain - number of residues in domain
    6. num_segments - number of individual segments in domain. 
    7. plddt - average pLDDT for domain (range from 0 to 100)
    8. num_helix_strand_turn - number of helix strand turns predicted by STRIDE
    9. num_helix - number of helices predicted by STRIDE
    10. num_strand - number of strands predicted by STRIDE
    11. num_helix_strand - number of helices and strands predicted by STRIDE
    12. num_turn - number of turns predicted by STRIDE
    13. proteome_id - proteome identifier in the format proteome-tax_id-<taxonID>-<shard>_v4 i.e. proteome-tax_id-67581-0_v4
    14. cath_label - CATH superfamily code if predicted, either a C.A.T.H. homologous superfamily or C.A.T. fold assignment. i.e. 3.40.50.300. Otherwise '-'
    15. cath_assignment_level - H for homologous superfamily assignment, T for fold level assignment. Otherwise '-'
    16. cath_assignment_method - Method used to assign a CATH label, either foldseek or foldclass. Otherwise '-'
    17. packing_density - metric used to determine globularity. A domain with packing_density >=10.333 and norm_rg below 0.356 is considered globular
    18. norm_rg - normalised radius of gyration. A domain with packing_density >=10.333 AND norm_rg below 0.356 is considered globular. 
    19. tax_common_name - Common name for organism
    20. tax_scientific_name - Scientific name for organism
    21. tax_lineage - Full taxonomic lineage.
"""


import pandas as pd
from download_seq import download_uniprot_sequence

with open("ted_365m.domain_summary.cath.globularity.taxid.tsv", "r") as f:
    lines = [next(f) for _ in range(1_000_000)]

print("Length of extracted lines:", len(lines))

df = pd.DataFrame([line.strip().split("\t") for line in lines])
df.columns = ["ted_id", "md5_domain", "consensus_level", "chopping", "nres_domain", "num_segments", "plddt", "num_helix_strand_turn", "num_helix", "num_strand", "num_helix_strand", "num_turn", "proteome_id", "cath_label", "cath_assignment_level", "cath_assignment_method", "packing_density", "norm_rg", "tax_common_name", "tax_scientific_name", "tax_lineage"]
df["uniprot_id"] = df["ted_id"].apply(lambda x: x.split("-")[1])
df['longest_domain_length'] = df["chopping"].apply(lambda x: int(x.split("-")[-1]) - int(x.split("-")[0]) + 1)
df['nres_domain'] = df['nres_domain'].astype(int)
df['ratio_longest_domain_length_to_nres_domain'] = df.apply(lambda x: int(x["nres_domain"]) / int(x["longest_domain_length"]) if int(x["longest_domain_length"]) > 0 else 0, axis=1)

new_df = df[["ted_id", "uniprot_id", "chopping", "nres_domain", "longest_domain_length", "ratio_longest_domain_length_to_nres_domain", "num_segments", "cath_label"]]
print(max(new_df["longest_domain_length"]), min(new_df["longest_domain_length"]))
print(max(new_df["nres_domain"]), min(new_df["nres_domain"]))
print(max(new_df["ratio_longest_domain_length_to_nres_domain"]), min(new_df["ratio_longest_domain_length_to_nres_domain"]))
new_df.to_csv("patched.ted.csv", index=False)


chopping_star_df = new_df.groupby("uniprot_id").apply(lambda x: " * ".join([f"{chopping} | {cath_label}" for chopping, cath_label in zip(x["chopping"], x["cath_label"])]))
chopping_star_df = chopping_star_df.reset_index()
chopping_star_df.columns = ["uniprot_id", "chopping_star"]
chopping_star_df.to_csv("chains.ted.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("patched.ted.csv")

plt.hist(
    df["longest_domain_length"], bins=100, 
    alpha=0.5, label="Longest domain length", color='blue'
)
plt.hist(
    df["nres_domain"], bins=100, 
    alpha=0.5, label="Number of residues in domain", color='orange'
)
plt.xlabel("Residue count")
plt.ylabel("Count")
plt.title("Distribution of domain and longest domain lengths")
plt.legend()
plt.savefig("overlaid_domain_lengths.png")
plt.close()